In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model_name="kiro", 
    temperature=0.7, 
    base_url="http://localhost:20128/v1", 
    api_key="sk-b99a23ccdde95cd0-3aa58d-8a494e39"
)

print(model.invoke("Write a short poem about the beauty of nature."))

content="Here's a short poem for you:\n\n---\n\n**Still Morning**\n\nThe fog sits low on borrowed hills,\na heron stands without a sound.\nLight comes slow the way it will,\nunbothered, even, spreading round.\n\nA creek runs cold through patient stone,\nthe pines hold still against the sky.\nNothing here is lost or gone —\njust the world, doing what worlds do, and I." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 759, 'total_tokens': 845, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'claude-sonnet-4.6', 'system_fingerprint': None, 'id': 'chatcmpl-1789462300292', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0a443-bd40-72e0-95f7-29cc68339cad-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 759, 'output_tokens': 86, 'total_tokens': 845, 'input_token_details': {}, 'output_token_details': {}}


In [7]:
res = model.invoke("berikan reasoning 15: 8?")
print(res.content)

15 ÷ 8 = **1.875**

Caranya:
- 8 × 1 = 8, sisa 7
- 70 ÷ 8 = 8, sisa 6 → dapat 0.8
- 60 ÷ 8 = 7, sisa 4 → dapat 0.07
- 40 ÷ 8 = 5, sisa 0 → dapat 0.005

Jadi 15 ÷ 8 = 1.875


In [8]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 1. Define the desired data structure using Pydantic
class Movie(BaseModel):
    answer: float = Field(description="The answer to the query")
    reasoning: str = Field(description="The reasoning behind the answer")

parser = JsonOutputParser(pydantic_object=Movie)
print("PROMPT PARSER AUOTOMATIS")
print(parser.get_format_instructions())

PROMPT PARSER AUOTOMATIS
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or M

In [13]:
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\nQuery: {query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

print(prompt.invoke({"query": "15: 8"}).text)

Answer the user query.
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Mar

In [ ]:
response = model.invoke("""
Answer the user query.
STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):
```
{"properties": {"answer": {"description": "The answer to the query", "title": "Answer", "type": "number"}, "reasoning": {"description": "The reasoning behind the answer", "title": "Reasoning", "type": "string"}}, "required": ["answer", "reasoning"]}
```
Query: 15: 8
""").content
print(response)

{"answer": 1.875, "reasoning": "The query '15: 8' represents division. 15 divided by 8 equals 1.875."}


TypeError: string indices must be integers, not 'str'

In [19]:
final_output = parser.invoke('''{"answer": 1.875, "reasoning": "The query '15: 8' represents division. 15 divided by 8 equals 1.875."}''')
print(final_output)
print(type(final_output))
print(final_output["answer"])

{'answer': 1.875, 'reasoning': "The query '15: 8' represents division. 15 divided by 8 equals 1.875."}
<class 'dict'>
1.875


In [21]:
# 4. Combine into a chain and invoke
chain = prompt | model | parser

result = chain.invoke({"query": "15: 8"})
print(result['answer'])

1.875
